# 02 — Предобработка: маски, патчи, train/val/test

Требует наличия скачанных снимков (`data/raw/sentinel2/sentinel2_<year>.tif`)
и контуров RGI (`data/rgi/`). Если данных пока нет — см. `_synthetic_smoke_test.py`
для проверки пайплайна на синтетических данных.


In [ ]:
import sys
sys.path.append('..')

import numpy as np
import geopandas as gpd
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt

from src import config, data_loader, preprocessing


## 2.1 Растеризация контуров RGI в маски

In [ ]:
rgi_path = config.DATA_RGI / 'rgi_study_area.shp'
rgi = gpd.read_file(rgi_path)
print(f"Контуров ледников в области: {len(rgi)}")

config.DATA_MASKS.mkdir(parents=True, exist_ok=True)

# Для лет 2020-2024 используем RGI как есть (свежие контуры),
# для более старых лет см. README про ручную корректировку в QGIS.
for year in [2020, 2021, 2022, 2023, 2024]:
    ref_path = config.DATA_RAW_SENTINEL2 / f'sentinel2_{year}.tif'
    if not ref_path.exists():
        print(f'  {year}: нет снимка, пропуск')
        continue
    out_path = config.DATA_MASKS / f'mask_{year}.tif'
    mask = preprocessing.rasterize_rgi_to_mask(rgi, ref_path, out_path)
    print(f'  {year}: маска сохранена, пикселей ледника = {mask.sum()}')


## 2.2 Загрузка снимков и масок, EDA

In [ ]:
sentinel_files = sorted(config.DATA_RAW_SENTINEL2.glob('sentinel2_20*.tif'))
print(f'Найдено снимков: {len(sentinel_files)}')

# Пример: посмотреть один снимок и его маску
example_year = 2024
img = data_loader.load_image(config.DATA_RAW_SENTINEL2 / f'sentinel2_{example_year}.tif')
mask = data_loader.load_mask(config.DATA_MASKS / f'mask_{example_year}.tif')

print('Снимок:', img.shape, 'Маска:', mask.shape)
print(f'Доля пикселей ледника: {mask.mean():.2%}')

from src import visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
visualization.show_rgb(img, ax=axes[0], title=f'Sentinel-2 {example_year} (RGB)')
visualization.show_mask(mask, ax=axes[1], title='RGI маска ледников')
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / 'eda_example.png', dpi=150)
plt.show()


## 2.3 Нарезка на патчи и сборка датасета

In [ ]:
pairs = []
for img_file in tqdm(sentinel_files):
    year = img_file.stem.split('_')[1]
    mask_file = config.DATA_MASKS / f'mask_{year}.tif'
    if not mask_file.exists():
        print(f'Нет маски для {year}, пропускаем')
        continue

    image = data_loader.load_image(img_file)
    mask = data_loader.load_mask(mask_file)
    pairs.append((image, mask))
    print(f'  {year}: снимок {image.shape}, ледник {mask.mean():.2%}')

X, y = preprocessing.build_dataset(pairs)
print(f'\nИтого патчей: {X.shape[0]}')
print(f'Форма патча: {X.shape[1:]}')
print(f'Баланс классов: {y.mean():.2%} пикселей — ледник')


## 2.4 Разделение train/val/test и сохранение

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = preprocessing.train_val_test_split(X, y)

print(f'Обучение:  {X_train.shape[0]} патчей')
print(f'Валидация: {X_val.shape[0]} патчей')
print(f'Тест:      {X_test.shape[0]} патчей')

config.DATA_PATCHES.mkdir(parents=True, exist_ok=True)
np.save(config.DATA_PATCHES / 'X_train.npy', X_train)
np.save(config.DATA_PATCHES / 'X_val.npy',   X_val)
np.save(config.DATA_PATCHES / 'X_test.npy',  X_test)
np.save(config.DATA_PATCHES / 'y_train.npy', y_train)
np.save(config.DATA_PATCHES / 'y_val.npy',   y_val)
np.save(config.DATA_PATCHES / 'y_test.npy',  y_test)

print('\nДанные сохранены в data/processed/patches/')
